# 05 — Results & Evaluation

**Project:** Predictive Analytics for Enterprise Streaming Acquisitions  
**Course:** CMPS 451 — Data Mining, Big Data & Analytics (Spring 2026)  
**Team:** 11

---

## Objectives
1. Compile and compare all model results
2. Generate confusion matrices for classification
3. Analyze feature importance from best models
4. Visualize K-Means clusters via PCA
5. Produce final business insights and recommendations

In [ ]:
# ── Fix PySpark path issue with spaces ──
import os
import sys
venv_scripts = os.path.dirname(sys.executable)
os.environ["PATH"] = f"{venv_scripts};{os.environ.get('PATH', '')}"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

# ── Imports ──
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.regression import (
    LinearRegressionModel, RandomForestRegressionModel, GBTRegressionModel
)
from pyspark.ml.classification import RandomForestClassificationModel
from pyspark.ml.clustering import KMeansModel
from pyspark.ml import Pipeline

plt.style.use('seaborn-v0_8-darkgrid')
matplotlib.rcParams.update({'font.size': 12, 'figure.figsize': (12, 7), 'figure.dpi': 150})

BASE_DIR = r"e:\CUFE\Spring_25\Big Data\Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")

# Fix Windows PySpark: set HADOOP_HOME
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
# Fix Windows PySpark: set HADOOP_HOME
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

    SparkSession.builder
    .appName("IMDb_Evaluation")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready.")

## 1. Regression Results Comparison

In [ ]:
# ── Load regression results ──
reg_df = pd.read_csv(os.path.join(RESULTS_DIR, "regression_results.csv"))

print("="*80)
print("REGRESSION MODEL COMPARISON")
print("="*80)
print(reg_df.to_string(index=False))

# Highlight best model
best_reg = reg_df.loc[reg_df['test_r2'].idxmax()]
print(f"\n🏆 Best Regression Model: {best_reg['model']}")
print(f"   Test R²: {best_reg['test_r2']:.4f}, Test RMSE: {best_reg['test_rmse']:.4f}")

In [ ]:
# ── Visualization: Regression model comparison ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (metric, title) in enumerate([
    ('rmse', 'RMSE (lower is better)'),
    ('mae', 'MAE (lower is better)'),
    ('r2', 'R² (higher is better)')
]):
    ax = axes[idx]
    x = range(len(reg_df))
    width = 0.25
    ax.bar([i - width for i in x], reg_df[f'train_{metric}'], width, label='Train', color='#3498db')
    ax.bar(x, reg_df[f'val_{metric}'], width, label='Validation', color='#e67e22')
    ax.bar([i + width for i in x], reg_df[f'test_{metric}'], width, label='Test', color='#2ecc71')
    ax.set_xticks(x)
    ax.set_xticklabels(reg_df['model'], rotation=25, ha='right', fontsize=9)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Regression Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '12_regression_comparison.png'))
plt.show()

## 2. Classification Results Comparison

In [ ]:
# ── Load classification results ──
cls_df = pd.read_csv(os.path.join(RESULTS_DIR, "classification_results.csv"))

print("="*80)
print("CLASSIFICATION MODEL COMPARISON")
print("="*80)
print(cls_df.to_string(index=False))

best_cls = cls_df.loc[cls_df['test_f1'].idxmax()]
print(f"\n🏆 Best Classification Model: {best_cls['model']}")
print(f"   Test Accuracy: {best_cls['test_accuracy']:.4f}, Test F1: {best_cls['test_f1']:.4f}")

In [ ]:
# ── Visualization: Classification model comparison ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (metric, title) in enumerate([
    ('accuracy', 'Accuracy'),
    ('f1', 'F1 Score')
]):
    ax = axes[idx]
    x = range(len(cls_df))
    width = 0.25
    ax.bar([i - width for i in x], cls_df[f'train_{metric}'], width, label='Train', color='#3498db')
    ax.bar(x, cls_df[f'val_{metric}'], width, label='Validation', color='#e67e22')
    ax.bar([i + width for i in x], cls_df[f'test_{metric}'], width, label='Test', color='#2ecc71')
    ax.set_xticks(x)
    ax.set_xticklabels(cls_df['model'], rotation=25, ha='right', fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Classification Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '13_classification_comparison.png'))
plt.show()

## 3. Confusion Matrix (Best Classifier)

In [ ]:
# ── Load best classifier and test data ──
with open(os.path.join(OUTPUT_DIR, "feature_columns.json")) as f:
    feat_meta = json.load(f)

df = spark.read.parquet(os.path.join(OUTPUT_DIR, "parquet", "features"))
feature_cols = feat_meta['all_feature_cols']
df = df.dropna(subset=feature_cols + ['averageRating', 'ratingTier'])

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)
pipeline = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df)
df_ml = pipeline_model.transform(df)
df_ml = df_ml.withColumnRenamed("averageRating", "label_reg")
df_ml = df_ml.withColumnRenamed("ratingTier", "label_cls")

_, _, test_df = df_ml.randomSplit([0.70, 0.15, 0.15], seed=42)

# Load best classifier (RF)
rf_cls = RandomForestClassificationModel.load(os.path.join(MODELS_DIR, "rf_classifier"))
preds = rf_cls.transform(test_df)

# Build confusion matrix
cm_data = (
    preds.groupBy("label_cls", "prediction")
    .count()
    .toPandas()
)

labels = [0, 1, 2]
cm = np.zeros((3, 3), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row['label_cls']), int(row['prediction'])] = int(row['count'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low (<5)', 'Medium (5-7)', 'High (≥7)'],
            yticklabels=['Low (<5)', 'Medium (5-7)', 'High (≥7)'],
            ax=ax)
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('Actual', fontsize=14)
ax.set_title('Confusion Matrix — Best Classifier (Random Forest)', fontsize=14, fontweight='bold')
plt.savefig(os.path.join(PLOTS_DIR, '14_confusion_matrix.png'))
plt.show()

## 4. Feature Importance (Random Forest)

In [ ]:
# ── Feature importance from best RF regressor ──
rf_reg = RandomForestRegressionModel.load(os.path.join(MODELS_DIR, "rf_regressor"))
importances = rf_reg.featureImportances.toArray()

feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

# Plot top 20
top_n = feat_imp.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(top_n)), top_n['importance'], color='#3498db', edgecolor='white')
ax.set_yticks(range(len(top_n)))
ax.set_yticklabels(top_n['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance', fontsize=14)
ax.set_title('Top 20 Feature Importances (Random Forest Regressor)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '15_feature_importance.png'))
plt.show()

print("\nTop 10 Features:")
print(feat_imp.head(10).to_string(index=False))

## 5. K-Means Cluster Visualization (PCA)

In [ ]:
# ── PCA for cluster visualization ──
kmeans_model = KMeansModel.load(os.path.join(MODELS_DIR, "kmeans"))

# Apply PCA to reduce to 2D
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(df_ml)
df_pca = pca_model.transform(df_ml)

# Add cluster labels
df_clustered = kmeans_model.transform(df_pca)

# Sample for plotting
sample_pdf = df_clustered.select("pca_features", "prediction", "label_reg").toPandas()
if len(sample_pdf) > 15000:
    sample_pdf = sample_pdf.sample(15000, random_state=42)

# Extract PCA components
sample_pdf['PC1'] = sample_pdf['pca_features'].apply(lambda x: float(x[0]))
sample_pdf['PC2'] = sample_pdf['pca_features'].apply(lambda x: float(x[1]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Left: Color by cluster
scatter1 = ax1.scatter(sample_pdf['PC1'], sample_pdf['PC2'],
                       c=sample_pdf['prediction'], cmap='Set1',
                       alpha=0.4, s=5)
ax1.set_xlabel('PC1', fontsize=14)
ax1.set_ylabel('PC2', fontsize=14)
ax1.set_title('K-Means Clusters (PCA Projection)', fontsize=14, fontweight='bold')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Right: Color by rating
scatter2 = ax2.scatter(sample_pdf['PC1'], sample_pdf['PC2'],
                       c=sample_pdf['label_reg'], cmap='RdYlGn',
                       alpha=0.4, s=5, vmin=1, vmax=10)
ax2.set_xlabel('PC1', fontsize=14)
ax2.set_ylabel('PC2', fontsize=14)
ax2.set_title('PCA Projection colored by Rating', fontsize=14, fontweight='bold')
plt.colorbar(scatter2, ax=ax2, label='Rating')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '16_kmeans_pca.png'))
plt.show()

## 6. Final Summary & Business Insights

In [ ]:
# ── Clustering results ──
with open(os.path.join(RESULTS_DIR, "clustering_results.json")) as f:
    clust_results = json.load(f)

print("="*80)
print("FINAL PROJECT SUMMARY")
print("="*80)

print("\n📊 REGRESSION (Predicting averageRating):")
print(reg_df[['model', 'test_rmse', 'test_mae', 'test_r2']].to_string(index=False))

print("\n📊 CLASSIFICATION (Predicting Rating Tier):")
print(cls_df[['model', 'test_accuracy', 'test_f1', 'test_precision', 'test_recall']].to_string(index=False))

print(f"\n📊 CLUSTERING (K-Means):")
print(f"   Best K: {clust_results['best_k']}")
print(f"   Silhouette Score: {clust_results['silhouette_scores'][str(clust_results['best_k'])]}")

print("\n" + "="*80)
print("BUSINESS INSIGHTS")
print("="*80)
print("""
1. DIRECTOR MATTERS: Director track record is one of the strongest predictors
   of a title's audience reception. Acquisition managers should prioritize
   titles from directors with consistently high-rated filmographies.

2. CAST QUALITY SIGNALS: Top-billed cast members' historical ratings provide
   a reliable signal. Titles featuring actors with strong track records
   tend to receive higher audience ratings.

3. GENRE STRATEGY: Certain genres (Documentary, Biography, Animation) tend
   to receive higher average ratings. Platforms should consider genre mix
   in their acquisition strategy.

4. LANGUAGE BIAS (⚠️ per instructor feedback): English-language titles
   dominate the dataset. Non-English titles may represent undervalued
   acquisition opportunities ("hidden gems").

5. VOTE COUNT RELIABILITY (⚠️ per instructor feedback): Titles with higher
   numVotes show more stable ratings. For acquisition decisions, prefer
   titles with sufficient audience sample sizes.

6. MICRO-GENRE CLUSTERS: K-Means clustering reveals distinct title segments
   with different characteristics. This can help platforms identify niche
   content categories for targeted audiences.
""")

In [ ]:
# ── Save comprehensive summary ──
summary = {
    "regression": reg_df.to_dict('records'),
    "classification": cls_df.to_dict('records'),
    "clustering": clust_results,
    "best_regression_model": best_reg['model'],
    "best_classification_model": best_cls['model'],
    "top_features": feat_imp.head(10).to_dict('records')
}

with open(os.path.join(RESULTS_DIR, "final_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n✅ Final summary saved to outputs/results/final_summary.json")
print("✅ All plots saved to outputs/plots/")
print("✅ All models saved to outputs/models/")
print("\n🎉 Project complete!")

In [ ]:
spark.stop()
print("Spark session stopped.")